# Notebook 8: AdaBoost — Adaptive Boosting

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 3 hours | **Topic:** Tree-Based Methods & Gradient Boosting

---

## Why This Matters

Random Forests win by **diversity** — many independent experts voting. AdaBoost wins by **focus** — a sequence of specialists, each one trained to fix the mistakes of all previous ones.

AdaBoost (Freund & Schapire, 1997) was the first practical boosting algorithm, won the Gödel Prize in 2003, and remains one of the most elegant ideas in machine learning. Understanding it unlocks intuition for all modern boosting methods: Gradient Boosting, XGBoost, LightGBM, and CatBoost are all variants of the same core insight.

For house price prediction, AdaBoost-style thinking is: "My first simple rule gets most houses right. But some houses are unusual — old Victorian downtown with a converted garage. Let me train specifically on those hard cases."

---

## Real-World Analogy First

Imagine a **tutoring program** for 800 students who must pass a "Is this house above/below median price?" exam:

- **Tutor 1** teaches everyone equally. After the session, 30% of students (240 students) still get questions wrong — mostly unusual edge cases.

- **Tutor 2** looks at Tutor 1's results. Those 240 struggling students get **more attention** (their weights increase). The other 560 students fade a bit to the background. Tutor 2 becomes an expert on the hard cases.

- **Tutor 3** looks at Tutor 2's results. The students Tutor 2 STILL couldn't help get even more attention.

- Final grade = **weighted average** of all tutors' assessments. Tutors who helped struggling students (low error rate) get more weight in the final grade.

The brilliant insight: each tutor is a **weak learner** (barely better than guessing), but the committee of tutors — each focusing on what the others missed — forms a **strong learner**.

## The AdaBoost Algorithm (Freund & Schapire, 1997)

### Setup
- Training set: $(x_1, y_1), \ldots, (x_n, y_n)$ where $y_i \in \{-1, +1\}$
- Weak learner: typically a **decision stump** (depth-1 tree — one question, two answers)
- $T$ = number of rounds (stumps to train)

### Algorithm

**Initialize** sample weights: $w_i^{(1)} = \frac{1}{n}$ for all $i$

**For $t = 1, 2, \ldots, T$:**

1. **Fit weak learner** $h_t$ to training data using weights $w^{(t)}$

2. **Weighted error:**
$$\varepsilon_t = \frac{\sum_{i=1}^{n} w_i^{(t)} \cdot \mathbf{1}[h_t(x_i) \neq y_i]}{\sum_{i=1}^{n} w_i^{(t)}}$$

3. **Learner weight** (higher $\alpha_t$ for better stumps):
$$\alpha_t = \frac{1}{2} \ln\left(\frac{1 - \varepsilon_t}{\varepsilon_t}\right)$$

4. **Update sample weights** (misclassified samples increase, correct samples decrease):
$$w_i^{(t+1)} \leftarrow w_i^{(t)} \cdot \exp\left(-\alpha_t \cdot y_i \cdot h_t(x_i)\right)$$

5. **Normalize:** $w_i^{(t+1)} \leftarrow \dfrac{w_i^{(t+1)}}{\sum_j w_j^{(t+1)}}$

**Final prediction:**
$$H(x) = \text{sign}\left(\sum_{t=1}^{T} \alpha_t \cdot h_t(x)\right)$$

### Weight Update Intuition

| Case | $y_i \cdot h_t(x_i)$ | $\exp(-\alpha_t \cdot y_i \cdot h_t(x_i))$ | Effect |
|---|---|---|---|
| Correct ($y_i = h_t$) | $+1$ | $e^{-\alpha_t} < 1$ | Weight **decreases** |
| Wrong ($y_i \neq h_t$) | $-1$ | $e^{+\alpha_t} > 1$ | Weight **increases** |

### Why It Works: Exponential Loss

AdaBoost minimizes the **exponential loss**:
$$L = \sum_{i=1}^{n} \exp\left(-y_i \cdot F(x_i)\right) \quad \text{where } F(x) = \sum_t \alpha_t h_t(x)$$

Samples that are consistently misclassified have large $F(x_i)$ in the wrong direction, so $\exp(-y_i F(x_i))$ is large — the algorithm concentrates its next step on reducing these.

**Theoretical guarantee:** Under the weak learner assumption ($\varepsilon_t \leq 0.5 - \gamma$ for some $\gamma > 0$), training error decreases as $\exp(-2\gamma^2 T)$ — exponentially fast.

In [ ]:
# ── Cell 1: Imports and global settings ──────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print("Libraries loaded.")
print("Task: Binary classification — is house price ABOVE (+1) or BELOW (−1) median?")

In [ ]:
# ── Cell 2: Generate binary house price dataset ───────────────────────────────
# Labels: +1 = above median price, -1 = below median price

np.random.seed(42)
n = 800

# Features
location_score    = np.random.uniform(1, 10, n)
sqft              = np.random.uniform(800, 4000, n)
age_years         = np.random.uniform(0, 60, n)
school_rating     = np.random.uniform(1, 10, n)
crime_rate        = np.random.uniform(0, 10, n)
distance_downtown = np.random.uniform(1, 30, n)
has_garage        = np.random.randint(0, 2, n).astype(float)
recent_renovation = np.random.randint(0, 2, n).astype(float)

# Price formula
price = (
    120_000
    + 15_000 * location_score
    + 80    * sqft
    - 1_500 * age_years
    + 8_000 * school_rating
    - 6_000 * crime_rate
    - 3_000 * distance_downtown
    + 15_000 * has_garage
    + 10_000 * recent_renovation
    + np.random.normal(0, 25_000, n)
)

# Binary label: +1 if above median, -1 if below
median_price = np.median(price)
y = np.where(price >= median_price, 1, -1)

feature_names = ['location_score', 'sqft', 'age_years', 'school_rating',
                 'crime_rate', 'distance_downtown', 'has_garage', 'recent_renovation']

X = np.column_stack([location_score, sqft, age_years, school_rating,
                     crime_rate, distance_downtown, has_garage, recent_renovation])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Dataset: {n} houses")
print(f"Median price: ${median_price:,.0f}")
print(f"Labels: +1 (above median) = {(y==1).sum()}, -1 (below) = {(y==-1).sum()}")
print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]}")
print(f"\nFeatures: {feature_names}")

# Quick baseline: majority class accuracy
majority_acc = max((y_train==1).mean(), (y_train==-1).mean())
print(f"\nBaseline (majority class) accuracy: {majority_acc:.3f}")

In [ ]:
# ── Cell 3: AdaBoost from Scratch ─────────────────────────────────────────────

class AdaBoostScratch:
    """
    AdaBoost classifier implementing the Freund-Schapire (1997) algorithm.
    
    Parameters
    ----------
    n_estimators : int
        Number of weak learners (decision stumps) to train.
    
    Requires labels y in {-1, +1}.
    """
    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators
        self.alphas = []       # learner weights α_t
        self.stumps = []       # fitted weak learners h_t
        self.errors = []       # weighted error ε_t at each round
        self.weight_history = []  # snapshot of sample weights

    def fit(self, X, y):
        """
        Fit AdaBoost on (X, y) where y ∈ {-1, +1}.
        """
        n = len(X)
        w = np.ones(n) / n   # Step 0: uniform weights

        for t in range(self.n_estimators):
            # Step 1: Fit a decision stump on weighted data
            stump = DecisionTreeClassifier(max_depth=1, random_state=t)
            stump.fit(X, y, sample_weight=w)
            y_pred = stump.predict(X)

            # Step 2: Weighted error  ε_t = Σ w_i * 1[h_t(x_i) ≠ y_i]
            err = np.sum(w * (y_pred != y)) / np.sum(w)
            err = np.clip(err, 1e-10, 1 - 1e-10)  # avoid log(0)

            # Step 3: Learner weight  α_t = 0.5 * ln((1-ε) / ε)
            alpha = 0.5 * np.log((1 - err) / err)

            # Step 4: Update sample weights
            #   w_i ← w_i * exp(-α_t * y_i * h_t(x_i))
            #   Correct predictions: y_i * h_t = +1 → exp(-α) → weight shrinks
            #   Wrong predictions:   y_i * h_t = -1 → exp(+α) → weight grows
            w = w * np.exp(-alpha * y * y_pred)

            # Step 5: Normalize
            w /= w.sum()

            # Store everything
            self.alphas.append(alpha)
            self.stumps.append(stump)
            self.errors.append(err)

            # Save weight snapshots at key rounds
            if t in [0, 4, 9, 19]:
                self.weight_history.append((t + 1, w.copy()))

        return self

    def predict(self, X):
        """Final prediction: sign(Σ α_t * h_t(x))."""
        votes = sum(
            alpha * stump.predict(X)
            for alpha, stump in zip(self.alphas, self.stumps)
        )
        return np.sign(votes)

    def staged_predict(self, X):
        """Yield predictions after each round — for convergence analysis."""
        running_votes = np.zeros(len(X))
        for alpha, stump in zip(self.alphas, self.stumps):
            running_votes += alpha * stump.predict(X)
            yield np.sign(running_votes)

    def staged_score(self, X, y):
        """Return accuracy at each round."""
        return [accuracy_score(y, pred) for pred in self.staged_predict(X)]


print("AdaBoostScratch class defined.")
print("Key implementation details:")
print("  - Weak learner: DecisionTreeClassifier(max_depth=1) — a decision stump")
print("  - Weight init: uniform w_i = 1/n")
print("  - Alpha formula: 0.5 * ln((1 - ε) / ε)")
print("  - Weight update: w_i *= exp(-α * y_i * h_t(x_i))")
print("  - Labels MUST be in {-1, +1}")

In [ ]:
# ── Cell 4: Train AdaBoost Scratch and Verify Against sklearn ─────────────────
N_ESTIMATORS = 100

# Train scratch
ada_scratch = AdaBoostScratch(n_estimators=N_ESTIMATORS)
ada_scratch.fit(X_train, y_train)

scratch_train_acc = accuracy_score(y_train, ada_scratch.predict(X_train))
scratch_val_acc   = accuracy_score(y_val,   ada_scratch.predict(X_val))

# Train sklearn AdaBoost
# Note: sklearn uses SAMME algorithm by default with DecisionTreeClassifier(max_depth=1)
ada_sklearn = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=N_ESTIMATORS,
    algorithm='SAMME',
    random_state=42
)
# sklearn expects y in {0, 1} or {-1, 1} — SAMME works with {-1, 1}
ada_sklearn.fit(X_train, y_train)

sklearn_train_acc = accuracy_score(y_train, ada_sklearn.predict(X_train))
sklearn_val_acc   = accuracy_score(y_val,   ada_sklearn.predict(X_val))

print("=" * 50)
print("SCRATCH vs SKLEARN VERIFICATION")
print("=" * 50)
print(f"{'':25} {'Scratch':>10} {'Sklearn':>10}")
print("-" * 50)
print(f"{'Train Accuracy':25} {scratch_train_acc:>10.4f} {sklearn_train_acc:>10.4f}")
print(f"{'Val Accuracy':25} {scratch_val_acc:>10.4f} {sklearn_val_acc:>10.4f}")
print()
diff = abs(scratch_val_acc - sklearn_val_acc)
if diff < 0.02:
    print(f"Results match within {diff:.4f} (small differences due to tie-breaking).")
else:
    print(f"Difference: {diff:.4f} — may be due to randomization in stumps.")

print(f"\nBaseline accuracy: {max((y_val==1).mean(), (y_val==-1).mean()):.4f}")
print(f"Our AdaBoost improves over baseline by: {scratch_val_acc - max((y_val==1).mean(), (y_val==-1).mean()):+.4f}")

In [ ]:
# ── Cell 5: Sample Weight Evolution ───────────────────────────────────────────
# Scatter plot: first 200 training samples.
# Size of dot proportional to current sample weight.
# Hard misclassified samples grow larger over rounds.

# Use the 2 most informative features for 2D visualization
feat_x_idx, feat_y_idx = 0, 1  # location_score, sqft
feat_x_name = feature_names[feat_x_idx]  # location_score
feat_y_name = feature_names[feat_y_idx]  # sqft

n_show = 200  # show first 200 training samples

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

colors = np.where(y_train[:n_show] == 1, '#2a9d8f', '#e76f51')

for ax, (round_num, weights) in zip(axes, ada_scratch.weight_history):
    # Scale weights for dot size (make visible)
    w_show = weights[:n_show]
    sizes = (w_show / w_show.max()) * 400  # max dot size = 400
    sizes = np.clip(sizes, 5, 400)  # minimum size for visibility

    sc = ax.scatter(
        X_train[:n_show, feat_x_idx],
        X_train[:n_show, feat_y_idx],
        s=sizes,
        c=colors,
        alpha=0.7,
        edgecolors='white',
        linewidth=0.4
    )
    
    ax.set_xlabel(feat_x_name)
    ax.set_ylabel(feat_y_name)
    ax.set_title(f'After Round {round_num}\n(dot size ∝ sample weight)', fontweight='bold')
    
    # Add legend
    legend_handles = [
        mpatches.Patch(color='#2a9d8f', label='Above median (+1)'),
        mpatches.Patch(color='#e76f51', label='Below median (−1)'),
    ]
    ax.legend(handles=legend_handles, loc='upper left', fontsize=8)
    
    # Annotate max-weight sample
    max_idx = np.argmax(w_show)
    ax.annotate(
        f'highest\nweight',
        xy=(X_train[max_idx, feat_x_idx], X_train[max_idx, feat_y_idx]),
        xytext=(X_train[max_idx, feat_x_idx] + 0.5, X_train[max_idx, feat_y_idx] + 200),
        fontsize=7,
        arrowprops=dict(arrowstyle='->', color='black', lw=1.2),
        color='black'
    )
    
    stats_text = f'max_w={w_show.max()*100:.2f}%\neff_n={1/(n*w_show**2).sum():.0f}'
    ax.text(0.98, 0.02, stats_text, transform=ax.transAxes,
            ha='right', va='bottom', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.8))

fig.suptitle('AdaBoost Sample Weight Evolution\n(hard examples become focal — their dots grow larger)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("Key observation: after many rounds, a few consistently-wrong samples")
print("dominate the weight distribution — these are the 'hardest' houses to classify.")

In [ ]:
# ── Cell 6: First 4 Decision Stump Boundaries ─────────────────────────────────
# Show what split each stump makes in 2D feature space
# (location_score vs sqft — the two most informative features)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# Build a mesh grid
x_min, x_max = X_train[:, feat_x_idx].min() - 0.5, X_train[:, feat_x_idx].max() + 0.5
y_min, y_max = X_train[:, feat_y_idx].min() - 50,  X_train[:, feat_y_idx].max() + 50
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 200),
    np.linspace(y_min, y_max, 200)
)

# We'll project the stump onto the 2D space using only these 2 features.
# Build 2D-only stumps for visualization.
vis_feats = [feat_x_idx, feat_y_idx]
X_train_2d = X_train[:, vis_feats]

# Re-run AdaBoost just on 2D features for visualization
np.random.seed(42)
n2 = len(X_train_2d)
w_vis = np.ones(n2) / n2
vis_stumps = []
vis_alphas = []

for t in range(4):
    st = DecisionTreeClassifier(max_depth=1, random_state=t)
    st.fit(X_train_2d, y_train, sample_weight=w_vis)
    yp = st.predict(X_train_2d)
    err = np.sum(w_vis * (yp != y_train)) / np.sum(w_vis)
    err = np.clip(err, 1e-10, 1 - 1e-10)
    alpha = 0.5 * np.log((1 - err) / err)
    w_vis = w_vis * np.exp(-alpha * y_train * yp)
    w_vis /= w_vis.sum()
    vis_stumps.append(st)
    vis_alphas.append(alpha)

for t, (ax, stump, alpha, err) in enumerate(zip(axes, vis_stumps, vis_alphas, ada_scratch.errors[:4])):
    # Decision boundary
    grid_input = np.c_[xx.ravel(), yy.ravel()]
    Z = stump.predict(grid_input).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.2, cmap=plt.cm.RdYlGn, levels=[-2, 0, 2])
    ax.contour(xx, yy, Z, colors='navy', linewidths=1.5, levels=[0])

    # Training points
    point_colors = np.where(y_train == 1, '#2a9d8f', '#e76f51')
    ax.scatter(X_train_2d[:, 0], X_train_2d[:, 1],
               c=point_colors, s=12, alpha=0.5, edgecolors='none')

    # Extract split feature and threshold from stump
    tree = stump.tree_
    split_feat_local = tree.feature[0]
    split_thresh      = tree.threshold[0]
    split_feat_name   = [feat_x_name, feat_y_name][split_feat_local]

    ax.set_xlabel(feat_x_name)
    ax.set_ylabel(feat_y_name)
    ax.set_title(
        f'Stump {t+1}: {split_feat_name} ≤ {split_thresh:.2f}\n'
        f'α={alpha:.3f}  ε={err:.3f}',
        fontweight='bold'
    )

legend_handles = [
    mpatches.Patch(color='#2a9d8f', label='Above median (+1)'),
    mpatches.Patch(color='#e76f51', label='Below median (−1)'),
]
axes[0].legend(handles=legend_handles, loc='upper right', fontsize=8)

fig.suptitle('First 4 AdaBoost Decision Stumps\n(2D projection: location_score vs sqft)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 7: Alpha Values Over Rounds ─────────────────────────────────────────
# Plot α_t for each round. Shows which stumps get more voting weight.

alphas = np.array(ada_scratch.alphas)
errors = np.array(ada_scratch.errors)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Alpha values
axes[0].plot(range(1, N_ESTIMATORS+1), alphas, color='#264653', linewidth=1.5, alpha=0.8)
axes[0].fill_between(range(1, N_ESTIMATORS+1), alphas, alpha=0.15, color='#264653')
axes[0].set_xlabel('Round (t)')
axes[0].set_ylabel('Alpha (α_t)')
axes[0].set_title('Learner Weights α_t over Rounds\n(higher = stump has more vote weight)', fontweight='bold')
axes[0].axhline(y=np.mean(alphas), color='red', linestyle='--', alpha=0.7,
                label=f'Mean α = {np.mean(alphas):.3f}')
axes[0].legend(fontsize=9)

# Error values
axes[1].plot(range(1, N_ESTIMATORS+1), errors, color='#e76f51', linewidth=1.5)
axes[1].axhline(0.5, color='black', linestyle='--', linewidth=1, label='Random = 0.5')
axes[1].fill_between(range(1, N_ESTIMATORS+1), errors, 0.5, alpha=0.15, color='#e76f51')
axes[1].set_xlabel('Round (t)')
axes[1].set_ylabel('Weighted Error (ε_t)')
axes[1].set_title('Stump Weighted Error ε_t over Rounds\n(as weights shift to hard samples, error ↑)', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 0.6)

# Alpha vs Error relationship (the formula)
eps_range = np.linspace(0.01, 0.49, 200)
alpha_curve = 0.5 * np.log((1 - eps_range) / eps_range)
axes[2].plot(eps_range, alpha_curve, color='#2a9d8f', linewidth=2.5, label='α = 0.5 ln((1-ε)/ε)')
axes[2].scatter(errors, alphas, c='#264653', s=15, alpha=0.5, zorder=5, label='Actual stumps')
axes[2].axvline(0.5, color='red', linestyle='--', alpha=0.7, label='ε = 0.5 → α = 0')
axes[2].axhline(0,   color='gray', linestyle='-', alpha=0.3)
axes[2].set_xlabel('Weighted Error ε')
axes[2].set_ylabel('Alpha α')
axes[2].set_title('Alpha vs Error Formula\nα → ∞ as ε → 0; α → 0 as ε → 0.5', fontweight='bold')
axes[2].legend(fontsize=9)
axes[2].set_xlim(0, 0.55)

plt.tight_layout()
plt.show()

print("Key observations:")
print(f"  Round 1 α: {alphas[0]:.4f}  (early rounds, easier examples, lower error → higher α)")
print(f"  Final α:   {alphas[-1]:.4f} (later rounds, harder examples, higher error → lower α)")
print(f"  Min error: {errors.min():.4f} at round {errors.argmin()+1}")
print(f"  Max error: {errors.max():.4f} at round {errors.argmax()+1}")
print(f"\n  As rounds progress, stumps face harder samples → error increases → α decreases")
print(f"  This is self-correcting: weak stumps get less vote weight automatically!")

In [ ]:
# ── Cell 8: Convergence — Train and Val Error vs Number of Rounds ─────────────

train_scores = ada_scratch.staged_score(X_train, y_train)
val_scores   = ada_scratch.staged_score(X_val,   y_val)

rounds = list(range(1, N_ESTIMATORS + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(rounds, train_scores, color='#264653', linewidth=2, label='Train accuracy')
axes[0].plot(rounds, val_scores,   color='#e9c46a', linewidth=2, label='Val accuracy')
axes[0].axhline(y=majority_acc, color='gray', linestyle=':', linewidth=1.5, label=f'Baseline ({majority_acc:.3f})')

best_round = np.argmax(val_scores) + 1
best_val   = max(val_scores)
axes[0].axvline(x=best_round, color='#e76f51', linestyle='--', alpha=0.8, linewidth=1.5,
                label=f'Best val round = {best_round}')
axes[0].scatter([best_round], [best_val], color='#e76f51', s=100, zorder=5)

axes[0].set_xlabel('Number of Rounds (T)')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('AdaBoost Convergence\n(train vs validation accuracy)', fontweight='bold')
axes[0].legend()
axes[0].set_ylim(max(0.5, min(train_scores) - 0.05), 1.01)

# Error rate plot
train_errors_curve = [1 - s for s in train_scores]
val_errors_curve   = [1 - s for s in val_scores]

axes[1].plot(rounds, train_errors_curve, color='#264653', linewidth=2, label='Train error')
axes[1].plot(rounds, val_errors_curve,   color='#e9c46a', linewidth=2, label='Val error')
axes[1].fill_between(rounds, train_errors_curve, val_errors_curve,
                     alpha=0.1, color='#e76f51', label='Generalization gap')
axes[1].set_xlabel('Number of Rounds (T)')
axes[1].set_ylabel('Error Rate')
axes[1].set_title('AdaBoost Error Curves\n(gap = generalization gap)', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Training accuracy at T={N_ESTIMATORS}: {train_scores[-1]:.4f}")
print(f"Val accuracy at T={N_ESTIMATORS}:      {val_scores[-1]:.4f}")
print(f"Best val accuracy: {best_val:.4f} at round {best_round}")
print()

# Check overfitting
if val_scores[-1] < val_scores[best_round - 1] - 0.005:
    print(f"Overfitting detected: val accuracy drops after round {best_round}.")
    print(f"  Use early stopping or reduce n_estimators to ~{best_round}.")
else:
    print("No significant overfitting at T=100. AdaBoost is well-regularized here.")
    print("(With more rounds, overfitting typically appears.)") 

In [ ]:
# ── Cell 9: Effect of Base Learner Depth ──────────────────────────────────────
# Compare AdaBoost with depth-1, depth-2, depth-3 base learners.
# Deeper trees = stronger base learners = faster convergence but more overfitting risk.

depths = [1, 2, 3]
depth_colors = ['#264653', '#2a9d8f', '#e9c46a']
N_ROUNDS = 100

results = {}  # depth -> (train_scores, val_scores)

for depth in depths:
    ada_d = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=depth),
        n_estimators=N_ROUNDS,
        algorithm='SAMME',
        random_state=42
    )
    ada_d.fit(X_train, y_train)

    tr = [accuracy_score(y_train, p) for p in ada_d.staged_predict(X_train)]
    vl = [accuracy_score(y_val,   p) for p in ada_d.staged_predict(X_val)]
    results[depth] = (tr, vl)
    print(f"Depth {depth}: final train={tr[-1]:.4f}, final val={vl[-1]:.4f}, best val={max(vl):.4f} @ round {np.argmax(vl)+1}")

rounds_arr = list(range(1, N_ROUNDS + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for depth, color in zip(depths, depth_colors):
    tr, vl = results[depth]
    axes[0].plot(rounds_arr, tr, color=color, linewidth=2, label=f'Train depth={depth}', linestyle='-')
    axes[1].plot(rounds_arr, vl, color=color, linewidth=2, label=f'Val depth={depth}', linestyle='-')

axes[0].set_xlabel('Rounds')
axes[0].set_ylabel('Train Accuracy')
axes[0].set_title('Training Accuracy\nby Base Learner Depth', fontweight='bold')
axes[0].legend()

axes[1].set_xlabel('Rounds')
axes[1].set_ylabel('Validation Accuracy')
axes[1].set_title('Validation Accuracy\nby Base Learner Depth', fontweight='bold')
axes[1].legend()
axes[1].axhline(y=majority_acc, color='gray', linestyle=':', linewidth=1.2, label='Baseline')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  Depth 1 (stumps):  slowest convergence, most regularized, classical AdaBoost")
print("  Depth 2-3 (trees): faster convergence, may overfit with many rounds")
print("  Deeper learners need FEWER rounds to match the same validation accuracy")

In [ ]:
# ── Cell 10: The Exponential Loss Visualization ───────────────────────────────
# Show: AdaBoost minimizes exp(-y * F(x))
# Compare with 0-1 loss, hinge loss, log loss

margin = np.linspace(-3, 3, 300)  # y * F(x)

exp_loss    = np.exp(-margin)
hinge_loss  = np.maximum(0, 1 - margin)
logistic_loss = np.log(1 + np.exp(-margin))
zero_one_loss = (margin < 0).astype(float)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(margin, zero_one_loss,   color='black', linewidth=2, linestyle=':', label='0-1 Loss (ideal, non-convex)')
ax.plot(margin, exp_loss,        color='#e76f51', linewidth=2.5, label='Exponential Loss (AdaBoost)')
ax.plot(margin, hinge_loss,      color='#2a9d8f', linewidth=2, linestyle='--', label='Hinge Loss (SVM)')
ax.plot(margin, logistic_loss,   color='#264653', linewidth=2, linestyle='-.', label='Log Loss (Logistic Reg)')

ax.axvline(0, color='gray', linestyle='-', alpha=0.4, linewidth=1)
ax.set_xlabel('Functional Margin: y · F(x)')
ax.set_ylabel('Loss')
ax.set_title('Loss Functions: AdaBoost minimizes Exponential Loss\n(positive margin = correct prediction)', fontweight='bold')
ax.set_ylim(-0.2, 5)
ax.set_xlim(-3, 3)
ax.legend()

# Annotate key regions
ax.annotate('correct\n(margin > 0)', xy=(1.5, 0.5), fontsize=9, color='darkgreen',
            ha='center', style='italic')
ax.annotate('wrong\n(margin < 0)', xy=(-1.5, 4), fontsize=9, color='darkred',
            ha='center', style='italic')

plt.tight_layout()
plt.show()

print("Key property of exponential loss:")
print("  - Heavily penalizes large negative margins (confidently wrong predictions)")
print("  - This is why AdaBoost's sample weight update focuses on hard misclassifications")
print("  - The weight w_i ∝ exp(-y_i * F(x_i)) is exactly the loss gradient")
print("  - Sensitivity to outliers: exp loss grows very fast for wrong predictions")

In [ ]:
# ── Cell 11: AdaBoost vs Random Forest Comparison ────────────────────────────
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest for comparison
rf_clf = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_clf.fit(X_train, y_train)

# AdaBoost with stumps
ada_final = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=N_ESTIMATORS,
    algorithm='SAMME',
    random_state=42
)
ada_final.fit(X_train, y_train)

print("=" * 55)
print("ADABOOST vs RANDOM FOREST")
print("=" * 55)
print(f"{'Method':<30} {'Train Acc':>10} {'Val Acc':>10}")
print("-" * 55)

methods = [
    ('Baseline (majority)', None, None),
    ('AdaBoost depth-1 (T=100)', ada_final, ada_final),
    ('AdaBoost depth-3 (T=100)',
     AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=3),
                        n_estimators=N_ESTIMATORS, algorithm='SAMME', random_state=42).fit(X_train, y_train),
     None
    ),
    ('Random Forest (N=100)', rf_clf, None),
]

for name, model, _ in methods:
    if model is None:
        tr_acc = majority_acc
        vl_acc = max((y_val==1).mean(), (y_val==-1).mean())
    else:
        tr_acc = accuracy_score(y_train, model.predict(X_train))
        vl_acc = accuracy_score(y_val,   model.predict(X_val))
    print(f"{name:<30} {tr_acc:>10.4f} {vl_acc:>10.4f}")

print("\nKey conceptual differences:")
print("  AdaBoost:      sequential, adaptive weights, stumps, can overfit with T")
print("  Random Forest: parallel, bootstrap sampling, deep trees, rarely overfits with T")

In [ ]:
# ── Cell 12: Noise Sensitivity Demo ───────────────────────────────────────────
# AdaBoost is sensitive to label noise because it focuses on hard examples,
# and noisy samples are "hard" by definition.

noise_levels = [0.0, 0.05, 0.10, 0.15, 0.20]
n_estimators_list = [10, 25, 50, 75, 100]

print("Effect of label noise on AdaBoost vs Random Forest:")
print(f"{'Noise Level':>12} {'AdaBoost T=100':>16} {'RF N=100':>12}")
print("-" * 44)

noise_results_ada = []
noise_results_rf  = []

for noise in noise_levels:
    # Flip labels randomly
    np.random.seed(42)
    y_noisy = y_train.copy()
    flip_mask = np.random.rand(len(y_noisy)) < noise
    y_noisy[flip_mask] = -y_noisy[flip_mask]

    # AdaBoost
    ada_noisy = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=100, algorithm='SAMME', random_state=42
    ).fit(X_train, y_noisy)
    ada_acc = accuracy_score(y_val, ada_noisy.predict(X_val))

    # Random Forest
    rf_noisy = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_noisy.fit(X_train, y_noisy)
    rf_acc = accuracy_score(y_val, rf_noisy.predict(X_val))

    noise_results_ada.append(ada_acc)
    noise_results_rf.append(rf_acc)
    print(f"{noise:>11.0%} {ada_acc:>16.4f} {rf_acc:>12.4f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot([n*100 for n in noise_levels], noise_results_ada, 'o-',
        color='#e76f51', linewidth=2, markersize=7, label='AdaBoost')
ax.plot([n*100 for n in noise_levels], noise_results_rf, 's-',
        color='#2a9d8f', linewidth=2, markersize=7, label='Random Forest')
ax.set_xlabel('Label Noise Level (%)')
ax.set_ylabel('Validation Accuracy')
ax.set_title('AdaBoost vs Random Forest: Sensitivity to Label Noise\n(both trained on noisy labels, evaluated on clean val set)',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print("\nConclusion: AdaBoost degrades more sharply with label noise.")
print("  Reason: it assigns high weight to consistently-wrong samples")
print("  (mislabeled samples are always wrong → they dominate training)")

## Summary: AdaBoost Properties

| Property | AdaBoost | Random Forest |
|---|---|---|
| **Training strategy** | Sequential (each round depends on previous) | Parallel (trees independent) |
| **Diversity source** | Adaptive sample weights | Bootstrap + random feature selection |
| **Base learner** | Weak (stumps) | Strong (deep trees) |
| **Overfitting with more estimators** | Can overfit (eventually) | Rare |
| **Noise sensitivity** | High (focuses on outliers/mislabeled) | Low (averages away noise) |
| **Training speed** | Sequential = slower | Parallel = faster |
| **Interpretability** | Moderate | Low |
| **Key hyperparameter** | n_estimators (T), learning rate | n_estimators, max_depth |

### AdaBoost as a Special Case of Gradient Boosting

AdaBoost can be viewed as **gradient descent in function space** with exponential loss. This insight (by Friedman, Hastie, and Tibshirani) led directly to Gradient Boosting Machines (GBM) — the topic of Notebook 9. The key generalization: instead of exponential loss, use any differentiable loss function, and instead of updating weights, fit a new tree to the residuals (pseudo-residuals = negative gradient).

**AdaBoost → Gradient Boosting → XGBoost → LightGBM**: each step generalizes or optimizes the previous one.

## Self-Check Questions

---

### Question 1
A stump has weighted error $\varepsilon_t = 0.49$. What is $\alpha_t$? What if $\varepsilon_t = 0.01$? What does high $\alpha_t$ mean?

<details>
<summary>Answer</summary>

**For $\varepsilon_t = 0.49$:**
$$\alpha_t = 0.5 \ln\left(\frac{1 - 0.49}{0.49}\right) = 0.5 \ln\left(\frac{0.51}{0.49}\right) = 0.5 \ln(1.0408) \approx 0.020$$

Very small — this stump barely beats random guessing (50% error) and gets almost no vote weight.

**For $\varepsilon_t = 0.01$:**
$$\alpha_t = 0.5 \ln\left(\frac{0.99}{0.01}\right) = 0.5 \ln(99) \approx 2.30$$

Very large — this stump is nearly perfect on the weighted data and gets a very strong vote.

**High $\alpha_t$ means:** this particular stump had a low weighted error rate on the current distribution of sample weights. It was very good at the specific examples that the previous stumps failed on. These examples get downweighted after this round (the model has "learned" them), and the next stump will face a new, harder distribution.

**Practical note:** As rounds increase, stumps typically face harder and harder distributions (the easy examples have been downweighted), so $\varepsilon_t$ tends to increase and $\alpha_t$ tends to decrease over rounds — which is why later stumps contribute less to the final vote.

</details>

---

### Question 2
AdaBoost can overfit with too many rounds, unlike Random Forest. Why does adding more stumps eventually hurt AdaBoost?

<details>
<summary>Answer</summary>

**The core reason: AdaBoost focuses on the hardest examples — including outliers and label noise.**

In the early rounds, AdaBoost correctly identifies genuinely difficult houses and concentrates on them. But eventually, it has classified all the "legitimately hard" examples correctly. What remains with high weight are:
1. **Mislabeled samples** (a house labeled above-median but actually below — maybe due to data error)
2. **Genuine outliers** (e.g., a tiny house in a great location that sold for an unusual price)
3. **True decision boundary ambiguities** (houses right on the median)

As the algorithm increasingly concentrates on these irreducible-error samples, it starts fitting noise patterns specific to the training set. The resulting stumps memorize quirks of these few high-weight samples that don't generalize.

**Contrast with Random Forest:** Each tree is trained on a bootstrap sample with equal weights. Adding more trees just adds more "average" estimators — overfitting one tree doesn't dominate because all trees vote equally. The ensemble variance decreases monotonically with more trees.

**Practical fix for AdaBoost:** Use early stopping (monitor validation accuracy), or add a learning rate parameter to shrink each stump's contribution: $F_t(x) = F_{t-1}(x) + \eta \cdot \alpha_t h_t(x)$.

</details>

---

### Question 3
Why must the weak learner do better than random ($\varepsilon_t < 0.5$) for AdaBoost to work? What happens mathematically when $\varepsilon_t = 0.5$?

<details>
<summary>Answer</summary>

**When $\varepsilon_t = 0.5$:**
$$\alpha_t = 0.5 \ln\left(\frac{1 - 0.5}{0.5}\right) = 0.5 \ln(1) = 0.5 \times 0 = 0$$

Alpha = 0 means this stump gets **zero vote weight** in the final prediction. It contributes nothing. The weight update becomes:
$$w_i \leftarrow w_i \times \exp(0) = w_i \times 1$$

Sample weights are unchanged, so the next stump trains on the exact same distribution and will again get error ≈ 0.5 and alpha = 0. The algorithm is stuck — it stops learning.

**When $\varepsilon_t > 0.5$:** $\alpha_t < 0$, which means the stump's predictions are actively inverted. This could theoretically still work (just flip the stump's output), but it means the stump is doing *worse* than random on the weighted distribution — which would mean the weighting scheme itself is broken.

**The theoretical guarantee:** If we can always find a weak learner with $\varepsilon_t \leq 0.5 - \gamma$ for some fixed $\gamma > 0$ (the "weak learning assumption"), then training error converges to zero at rate $\exp(-2\gamma^2 T)$. The question of whether such a learner always exists for the reweighted distributions is deep — Freund & Schapire proved that the boosting algorithm itself (the adaptive weighting) **guarantees** it, given that the base learner can always find something better than random on any weighted distribution.

</details>

In [ ]:
# ── Cell 13: Numerical Self-Check — Verify Alpha Formula ─────────────────────
# Interactive calculation: compute α for different ε values

print("Alpha values for different weighted error rates:")
print(f"{'ε (error)':>12} {'α (weight)':>12} {'Interpretation'}")
print("-" * 65)

test_epsilons = [0.01, 0.05, 0.10, 0.20, 0.30, 0.40, 0.45, 0.49, 0.499, 0.5]

for eps in test_epsilons:
    if eps < 0.5:
        alpha = 0.5 * np.log((1 - eps) / eps)
        if alpha > 2:
            interp = "Excellent stump — dominates the vote"
        elif alpha > 1:
            interp = "Strong stump — high vote weight"
        elif alpha > 0.2:
            interp = "Decent stump — moderate weight"
        else:
            interp = "Weak stump — barely above random"
    else:
        alpha = 0.0
        interp = "← ZERO weight: no better than coin flip!"

    print(f"{eps:>12.3f} {alpha:>12.4f}  {interp}")

print("\nOur trained stumps:")
print(f"  Round 1: ε={ada_scratch.errors[0]:.4f} → α={ada_scratch.alphas[0]:.4f}")
print(f"  Round 10: ε={ada_scratch.errors[9]:.4f} → α={ada_scratch.alphas[9]:.4f}")
print(f"  Round 50: ε={ada_scratch.errors[49]:.4f} → α={ada_scratch.alphas[49]:.4f}")
print(f"  Round 100: ε={ada_scratch.errors[99]:.4f} → α={ada_scratch.alphas[99]:.4f}")

In [ ]:
# ── Cell 14: Final Summary Dashboard ─────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[0, 2])
ax4 = fig.add_subplot(gs[1, :])

# 1. Alpha values
ax1.plot(range(1, N_ESTIMATORS+1), ada_scratch.alphas, color='#264653', linewidth=1.5)
ax1.set_title('Learner Weights α_t', fontweight='bold')
ax1.set_xlabel('Round t')
ax1.set_ylabel('α_t')

# 2. Error values
ax2.plot(range(1, N_ESTIMATORS+1), ada_scratch.errors, color='#e76f51', linewidth=1.5)
ax2.axhline(0.5, color='black', linestyle='--', linewidth=1, alpha=0.7)
ax2.set_title('Weighted Error ε_t', fontweight='bold')
ax2.set_xlabel('Round t')
ax2.set_ylabel('ε_t')
ax2.set_ylim(0, 0.6)

# 3. Val accuracy curves for all depths
for depth, color in zip(depths, depth_colors):
    _, vl = results[depth]
    ax3.plot(range(1, N_ROUNDS+1), vl, color=color, linewidth=2, label=f'depth={depth}')
ax3.axhline(majority_acc, color='gray', linestyle=':', linewidth=1.2)
ax3.set_title('Val Accuracy by Depth', fontweight='bold')
ax3.set_xlabel('Rounds')
ax3.set_ylabel('Accuracy')
ax3.legend(fontsize=8)

# 4. Noise sensitivity
ax4.plot([n*100 for n in noise_levels], noise_results_ada, 'o-',
         color='#e76f51', linewidth=2, markersize=8, label='AdaBoost')
ax4.plot([n*100 for n in noise_levels], noise_results_rf, 's-',
         color='#2a9d8f', linewidth=2, markersize=8, label='Random Forest')
ax4.set_xlabel('Label Noise Level (%)')
ax4.set_ylabel('Validation Accuracy')
ax4.set_title('Noise Robustness: AdaBoost vs Random Forest', fontweight='bold')
ax4.legend()

fig.suptitle('AdaBoost Summary Dashboard', fontsize=14, fontweight='bold')
plt.show()

print("\n" + "=" * 55)
print("ADABOOST: KEY TAKEAWAYS")
print("=" * 55)
print("1. Builds a strong classifier from many weak learners")
print("2. Each stump focuses on the previous ones' mistakes")
print("3. Better stumps (lower error) get higher vote weight")
print("4. α_t → 0 as ε_t → 0.5 (prevents useless stumps from polluting)")
print("5. Sensitive to label noise (hard samples = mislabeled samples)")
print("6. Foundation for Gradient Boosting, XGBoost, LightGBM")

## Key Takeaways

1. **AdaBoost is sequential:** Each new weak learner is explicitly trained to fix the errors of all previous ones. This is the opposite of Random Forest's independence.

2. **Sample weights are the feedback mechanism:** Misclassified samples get higher weight → next stump focuses on them. This is the "adaptive" in AdaBoost.

3. **Alpha is automatic regularization:** Stumps with near-random performance ($\varepsilon \approx 0.5$) get near-zero weight ($\alpha \approx 0$). The algorithm self-regulates weak contributions.

4. **The weak learner assumption is key:** AdaBoost requires each stump to do *slightly* better than random. If the problem has no signal, AdaBoost degrades to random guessing.

5. **Noise is AdaBoost's enemy:** Mislabeled samples are persistently misclassified → they accumulate high weights → the algorithm wastes capacity on irreducible noise. Random Forests average this away.

6. **Exponential loss drives the math:** The sample weight update $w_i \propto \exp(-y_i F(x_i))$ is exactly the gradient of exponential loss. This connects AdaBoost to Gradient Boosting.

---

**Next:** Notebook 9 — Gradient Boosting Machines: The general framework that unifies AdaBoost and extends it to any loss function